In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spacy #NUEVO
import re
import unicodedata

from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import FunctionTransformer



In [2]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 55.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

def quitar_tildes(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

dataset = 'news_reducido.csv'
data = pd.read_csv(dataset, on_bad_lines='skip')
textos_crudos = data['text'].fillna('').astype(str).tolist()

# 4. Preprocesamiento en dos fases optimizadas
textos_limpios = []


textos_sin_ruido = [re.sub(r'\S+@\S+', '', re.sub(r'http\S+|www\S+|https\S+', '', t, flags=re.MULTILINE)) for t in textos_crudos]

for doc in nlp.pipe(textos_sin_ruido, batch_size=100):
    lemas = []
    for token in doc:
        if (not token.is_punct and
            not token.is_space and
            not token.is_stop and
            not token.like_num and
            len(token.text) > 2):

            lema_limpio = quitar_tildes(token.lemma_.lower())

            lema_limpio = re.sub(r'[^a-zñ]', '', lema_limpio)

            if lema_limpio:
                lemas.append(lema_limpio)

    textos_limpios.append(" ".join(lemas))

X = np.array(textos_limpios)


In [4]:
# Ahora, procesamos las etiquetas, para cada clase, le asignamos un valor numérico entre 0 y el número de clases
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla1

enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1) # DOCUMENTOS Y LE ASOCIO LA CATEGORIA

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
skf.get_n_splits(X, y)

# Definir aquí los pipelines necesarios para cada problema (clustering, clasificación, etc.)
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=4, random_state=semilla)) #Preguntar los randoms state
])

text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=4, random_state=semilla)) #Preguntar el random_state
])

text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=4, random_state=semilla))
])

text_kmeans_tfidf_mezcla_gausiana = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianMixture(n_components=4, random_state=semilla, covariance_type='spherical'))
])

text_kmeans = text_kmeans_binario

In [5]:
accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):

    fit_clustering = True
    # Clustering
    if fit_clustering:
        # Entrenamiento
        text_kmeans.fit(X[tra])
        # Test
        labels = text_kmeans.predict(X[tst])
        # Calculo de metricas
        acc = np.mean(labels == y[tst])
        print(acc)
        accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

0.225
0.22
0.3335
0.287
0.353
Precision media = 0.28369999999999995


In [6]:
dataset = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(dataset, on_bad_lines='skip')
# Nos quedamos con el texto (puedes quedarte con más información si quieres)
X = data['text'].fillna('').astype(str).to_numpy() # Ponco con un espacio los espacio nulos

In [7]:
# Ahora, procesamos las etiquetas, para cada clase, le asignamos un valor numérico entre 0 y el número de clases
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla1

enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1)

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
skf.get_n_splits(X, y)

# Definir aquí los pipelines necesarios para cada problema (clustering, clasificación, etc.)
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=4, random_state=semilla)) #Preguntar los randoms state
])

text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=4, random_state=semilla)) #Preguntar el random_state
])

text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=4, random_state=semilla))
])

text_kmeans_tfidf_mezcla_gausiana = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianMixture(n_components=4, random_state=semilla, covariance_type='spherical'))
])


text_sgd = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_kmeans = text_kmeans_tfidf_mezcla_gausiana

In [ ]:
accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):

    fit_clustering = True
    # Clustering
    if fit_clustering:
        # Entrenamiento
        text_kmeans.fit(X[tra])
        # Test
        labels = text_kmeans.predict(X[tst])
        # Calculo de metricas
        acc = np.mean(labels == y[tst])
        print(acc)
        accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')